# About

This notebook is for testing if the dataset cleaning was proper or not, as I encountered that a significant amount of data was removed from the dataset after cleaning.

## Note

This file is updated everytime I fix an issue, so actual issues won't be visible after the fix. If you are curious about that, consider reading development log in the docs folder.

## Imports

In [1]:
import pandas as pd
from src.core.config import config_loader
from src.preprocessing.preprocessing import PreprocessingPipeline
from src.dataset.loader import load_cicids2017_dataset
from src.preprocessing.cleaning import initial_cleanup
from pathlib import Path

## Loading Dataset

In [2]:
dataset_dir = '../dataset'

df = load_cicids2017_dataset(dataset_dir)
cleaning_cfg = config_loader('../config/preprocessing/cleaning.yaml')

In [3]:
df = initial_cleanup(df, cleaning_cfg)

## Validation & Cleaning

In [5]:
sp = Path('../config/preprocessing/validation_shema.yaml')
schema_cfg = config_loader(sp)

rp = Path('../config/preprocessing/validation_rules.yaml')
rules_cfg = config_loader(rp)

cleaning_path = Path('../config/preprocessing/cleaning.yaml')
cleaning_cfg = config_loader(cleaning_path)

preprocess = PreprocessingPipeline(schema_cfg, rules_cfg, cleaning_cfg)

preprocess.validate(df)
df_cleaned = preprocess.clean(df)

preprocess.validate(df_cleaned)
df_cleaned_final = preprocess.clean(df_cleaned)

preprocess.validate(df_cleaned_final)

print(preprocess.validation_result[f'validation_result_{len(preprocess.validation_result)}'])

Validation Result
----------------------
Repairable: 0
Non-repairable: 0
NaN*: 0
Inf*: 0
Negative*: 0
Out Of range*: 0
Rule violators: {'R001': np.int64(0), 'R002': np.int64(0), 'R003': np.int64(0), 'R004': np.int64(0), 'R005': np.int64(0), 'R006': np.int64(0), 'R007': np.int64(0), 'R008': np.int64(0), 'R009': np.int64(0), 'R010': np.int64(0), 'R011': np.int64(0), 'R012': np.int64(0), 'R013': np.int64(0), 'R014': np.int64(0), 'R015': np.int64(0), 'R016': np.int64(0), 'R017': np.int64(0), 'R018': np.int64(0), 'R019': np.int64(0)}
----------------------
NOTE: those marked with * are showing individual cell value count, not row.
Just so you know, a single row can have multiple NaN, Inf, Negative etc. values.


## Comparison

In [6]:
df_comparison = pd.DataFrame({
    'Original': [],
    'First Clean': [],
    'First Remove': [],
    'First remove percent': [],
    'Second Remove': [],
    'Second Removed': [],
    'Second Removed Percent': []
})

df_comparison['Original'] = df['Label'].value_counts()
df_comparison['First Clean'] = df_cleaned['Label'].value_counts()
df_comparison['First Remove'] = df['Label'].value_counts() - df_cleaned['Label'].value_counts()
df_comparison['First remove percent'] = 100 -(df_cleaned['Label'].value_counts() / df['Label'].value_counts()) * 100
df_comparison['Second Clean'] = df_cleaned_final['Label'].value_counts()
df_comparison['Second Remove'] = df_cleaned['Label'].value_counts() - df_cleaned_final['Label'].value_counts()
df_comparison['Second remove percent'] =  100 - (df_cleaned_final['Label'].value_counts() / df_cleaned['Label'].value_counts()) * 100

In [7]:
comp_path = Path('../out/data/')
comp_path.mkdir(parents=True, exist_ok=True)
df_comparison.to_csv(Path(comp_path, 'df_comparison.csv'))

In [8]:
df_comparison

,Original,First Clean,First Remove,First remove percent,Second Remove,Second Removed,Second Removed Percent,Second Clean,Second remove percent
Label,,,,,,,,,
BENIGN,2096484,2088143,8341,0.397857,0,NaN,NaN,2088143,0.0
DoS Hulk,172849,161713,11136,6.442618,0,NaN,NaN,161713,0.0
DDoS,128016,124411,3605,2.816054,0,NaN,NaN,124411,0.0
PortScan,90819,90561,258,0.284082,0,NaN,NaN,90561,0.0
DoS GoldenEye,10286,8945,1341,13.037138,0,NaN,NaN,8945,0.0
FTP-Patator,5933,5927,6,0.101129,0,NaN,NaN,5927,0.0
DoS slowloris,5385,4822,563,10.454968,0,NaN,NaN,4822,0.0
DoS Slowhttptest,5228,4830,398,7.612854,0,NaN,NaN,4830,0.0
SSH-Patator,3219,3217,2,0.062131,0,NaN,NaN,3217,0.0


From the above I can say for certain that the cleaning  was either wrong or too aggressive and removed a lot of data.

In [9]:
pd.set_option("display.float_format", "{:.2f}".format)

In [10]:
df['Active Max'].describe()

count     2522362.00
mean       171910.44
std       1085243.03
min             0.00
25%             0.00
50%             0.00
75%             0.00
max     110000000.00
Name: Active Max, dtype: float64

In [11]:
df['Idle Max'].describe()

count     2522362.00
mean      9757716.42
std      25610668.42
min             0.00
25%             0.00
50%             0.00
75%             0.00
max     120000000.00
Name: Idle Max, dtype: float64

In [12]:
df['Flow Duration'].describe()

count     2522362.00
mean     16581323.77
std      35224259.61
min           -13.00
25%           208.00
50%         50577.00
75%       5329717.25
max     119999998.00
Name: Flow Duration, dtype: float64

In [13]:
violations = df[
    (df["Active Max"] > df["Flow Duration"]) |
    (df["Idle Max"] > df["Flow Duration"])
]

In [14]:
violations["Label"].value_counts()

Label
DoS Hulk            10982
BENIGN               4295
DDoS                 3584
DoS GoldenEye        1336
DoS slowloris         563
DoS Slowhttptest      398
PortScan              133
Name: count, dtype: int64

In [15]:
ActiveRatio = df["Active Max"] / df["Flow Duration"]

IdleRatio = df["Idle Max"] / df["Flow Duration"]

In [16]:
ActiveRatio.describe()

count   2520798.00
mean          0.00
std           0.02
min           0.00
25%           0.00
50%           0.00
75%           0.00
max           0.96
dtype: float64

In [17]:
IdleRatio.describe()

count   2520798.00
mean          0.16
std           0.34
min           0.00
25%           0.00
50%           0.00
75%           0.00
max           1.00
dtype: float64

In [18]:
r14 = preprocess.rules_result['rules_result_1'].violators['R014']

df_r14 = df.loc[list(r14)][['Flow Duration', 'Active Max', 'Idle Max']]

In [19]:
df_r14[~ (df_r14['Flow Duration'] < df_r14['Active Max']) & ~ (df_r14['Flow Duration'] < df_r14['Idle Max'])]

,Flow Duration,Active Max,Idle Max


In [20]:
df.loc[list(r14)]['Label'].value_counts()

Label
DoS Hulk            10982
BENIGN               4295
DDoS                 3584
DoS GoldenEye        1336
DoS slowloris         563
DoS Slowhttptest      398
PortScan              133
Name: count, dtype: int64

In [21]:
web_brute_force_df = df[df['Label'] == 'Web Attack � Brute Force'] #3
web_xss_df = df[df['Label'] == 'Web Attack � XSS'] #4
web_sqli_df = df[df['Label'] == 'Web Attack � SQL Injection'] #5

In [22]:
preprocess.validate(web_brute_force_df)
preprocess.validate(web_xss_df)
preprocess.validate(web_sqli_df)

(<src.preprocessing.result.ValidationResult at 0x1d288694830>,
 <src.preprocessing.result.SchemaResult at 0x1d2873cbe30>)

In [23]:
print(preprocess.rules_result['rules_result_4'])
print(preprocess.schema_result['schema_result_4'])

R001: 0
R002: 0
R003: 0
R004: 0
R005: 0
R006: 0
R007: 0
R008: 0
R009: 0
R010: 0
R011: 0
R012: 0
R013: 0
R014: 0
R015: 0
R016: 0
R017: 0
R018: 0
R019: 0

Schema Validation Result
-----------------
NaN:  0
Inf:  0
Negative:  0
Out of range values:  0
-----------------
NOTE: These are number of cell values not rows.


In [24]:
print(preprocess.rules_result['rules_result_5'])
print(preprocess.schema_result['schema_result_5'])

R001: 0
R002: 0
R003: 0
R004: 0
R005: 0
R006: 0
R007: 0
R008: 0
R009: 0
R010: 0
R011: 0
R012: 0
R013: 0
R014: 0
R015: 0
R016: 0
R017: 0
R018: 0
R019: 0

Schema Validation Result
-----------------
NaN:  0
Inf:  0
Negative:  0
Out of range values:  0
-----------------
NOTE: These are number of cell values not rows.


In [25]:
print(preprocess.rules_result['rules_result_6'])
print(preprocess.schema_result['schema_result_6'])

R001: 0
R002: 0
R003: 0
R004: 0
R005: 0
R006: 0
R007: 0
R008: 0
R009: 0
R010: 0
R011: 0
R012: 0
R013: 0
R014: 0
R015: 0
R016: 0
R017: 0
R018: 0
R019: 0

Schema Validation Result
-----------------
NaN:  0
Inf:  0
Negative:  0
Out of range values:  0
-----------------
NOTE: These are number of cell values not rows.


In [26]:
for label in sorted(df["Label"].unique()):
    print([hex(ord(c)) for c in label])

['0x42', '0x45', '0x4e', '0x49', '0x47', '0x4e']
['0x42', '0x6f', '0x74']
['0x44', '0x44', '0x6f', '0x53']
['0x44', '0x6f', '0x53', '0x20', '0x47', '0x6f', '0x6c', '0x64', '0x65', '0x6e', '0x45', '0x79', '0x65']
['0x44', '0x6f', '0x53', '0x20', '0x48', '0x75', '0x6c', '0x6b']
['0x44', '0x6f', '0x53', '0x20', '0x53', '0x6c', '0x6f', '0x77', '0x68', '0x74', '0x74', '0x70', '0x74', '0x65', '0x73', '0x74']
['0x44', '0x6f', '0x53', '0x20', '0x73', '0x6c', '0x6f', '0x77', '0x6c', '0x6f', '0x72', '0x69', '0x73']
['0x46', '0x54', '0x50', '0x2d', '0x50', '0x61', '0x74', '0x61', '0x74', '0x6f', '0x72']
['0x48', '0x65', '0x61', '0x72', '0x74', '0x62', '0x6c', '0x65', '0x65', '0x64']
['0x49', '0x6e', '0x66', '0x69', '0x6c', '0x74', '0x72', '0x61', '0x74', '0x69', '0x6f', '0x6e']
['0x50', '0x6f', '0x72', '0x74', '0x53', '0x63', '0x61', '0x6e']
['0x53', '0x53', '0x48', '0x2d', '0x50', '0x61', '0x74', '0x61', '0x74', '0x6f', '0x72']
['0x57', '0x65', '0x62', '0x20', '0x41', '0x74', '0x74', '0x61', '0x